# BS-RoFormer - Audio Source Separation

This notebook demonstrates how to use BS-RoFormer for multi-stem audio source separation.

**Features:**
- Separate up to 6 stems: vocals, drums, bass, guitar, piano, other
- High-quality source separation using Band-Split transformer architecture
- Multiple pre-trained models available (vocal extraction, instrumental, dereverb)

## 1. Installation

In [ ]:
# Install bs-roformer-infer package
!pip install -q git+https://github.com/openmirlab/bs-roformer-infer.git

# TODO: After publishing to PyPI, use:
# !pip install -q bs-roformer-infer

In [ ]:
# Verify installation
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

## 2. Download Model

We'll use the **BS-RoFormer-SW** model by jarredou, which is excellent for 6-stem source separation (vocals, drums, bass, guitar, piano, other).

In [ ]:
# Create directories
!mkdir -p models
!mkdir -p input_songs
!mkdir -p outputs

In [ ]:
# Download model using the built-in downloader
!bs-roformer-download --model roformer-model-bs-roformer-sw-by-jarredou --output-dir models

print("\nModel and config ready!")
!ls -lh models/roformer-model-bs-roformer-sw-by-jarredou/

## 3. Upload Your Audio File

Upload a `.wav` file to separate into multiple stems.

In [ ]:
# Option 1: Upload from your computer
from google.colab import files

print("Upload your audio file (.wav format):")
uploaded = files.upload()

# Move uploaded file to input folder
for filename in uploaded.keys():
    !mv "{filename}" input_songs/
    print(f"Moved {filename} to input_songs/")

In [ ]:
# Option 2: Use a sample audio (uncomment to use)
# !wget -q -O input_songs/sample.wav "YOUR_AUDIO_URL_HERE"

In [ ]:
# Check input files
!ls -lh input_songs/

## 4. Run Source Separation

Select your preferred method and run the separation:

In [ ]:
#@title Select Inference Method { display-mode: "form" }
#@markdown Choose which method to use for audio separation:

inference_method = "CLI (Command Line)" #@param ["CLI (Command Line)", "Python API"]

print(f"Selected method: {inference_method}")

In [ ]:
#@title Run Source Separation { display-mode: "form" }
#@markdown Click the play button to run separation with your selected method.

if inference_method == "CLI (Command Line)":
    # ============================================
    # Option A: CLI (Command Line)
    # ============================================
    print("Running with CLI...\n")
    !bs-roformer-infer \
        --config_path models/roformer-model-bs-roformer-sw-by-jarredou/BS-Rofo-SW-Fixed.yaml \
        --model_path models/roformer-model-bs-roformer-sw-by-jarredou/BS-Rofo-SW-Fixed.ckpt \
        --input_folder ./input_songs \
        --store_dir ./outputs

else:
    # ============================================
    # Option B: Python API
    # ============================================
    print("Running with Python API...\n")
    
    import torch
    import yaml
    import soundfile as sf
    import numpy as np
    from pathlib import Path
    from ml_collections import ConfigDict
    from bs_roformer.utils import get_model_from_config, demix_track
    from bs_roformer.inference import SafeLoaderWithTuple

    # Load model
    print("Loading model...")
    config_path = "models/roformer-model-bs-roformer-sw-by-jarredou/BS-Rofo-SW-Fixed.yaml"
    model_path = "models/roformer-model-bs-roformer-sw-by-jarredou/BS-Rofo-SW-Fixed.ckpt"
    
    with open(config_path) as f:
        config = ConfigDict(yaml.load(f, Loader=SafeLoaderWithTuple))

    model = get_model_from_config("bs_roformer", config)
    model.load_state_dict(
        torch.load(model_path, map_location="cpu")
    )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    model.eval()
    print(f"Model loaded on {device}\n")

    # Process files
    input_folder = Path("input_songs")
    output_folder = Path("outputs")
    output_folder.mkdir(exist_ok=True)

    for audio_path in input_folder.glob("*.wav"):
        print(f"Processing: {audio_path.name}")
        
        # Load audio
        mix, sr = sf.read(audio_path)
        original_mono = len(mix.shape) == 1
        if original_mono:
            mix = np.stack([mix, mix], axis=-1)
        
        # Convert to tensor
        mixture = torch.tensor(mix.T, dtype=torch.float32)
        
        # Run separation
        with torch.no_grad():
            result, _ = demix_track(config, model, mixture, device)
        
        # Save all stems
        stem_name = audio_path.stem
        for instrument, audio in result.items():
            output = audio.T
            if original_mono:
                output = output[:, 0]
            output_path = output_folder / f"{stem_name}_{instrument}.wav"
            sf.write(output_path, output, sr, subtype="FLOAT")
            print(f"  Saved: {output_path}")
        
        # Save instrumental (original - vocals)
        if "vocals" in result:
            vocals = result["vocals"].T
            if original_mono:
                vocals = vocals[:, 0]
                mix_mono = mix[:, 0]
            else:
                mix_mono = mix
            instrumental = mix_mono - vocals
            instrumental_path = output_folder / f"{stem_name}_instrumental.wav"
            sf.write(instrumental_path, instrumental, sr, subtype="FLOAT")
            print(f"  Saved: {instrumental_path}")

    print("\nDone!")

## 5. Check Output Files

In [ ]:
# Check output files
!ls -lh outputs/

## 6. Listen to Results

In [ ]:
import IPython.display as ipd
from pathlib import Path

output_dir = Path("outputs")

# Find and display all output files
for audio_file in sorted(output_dir.glob("*.wav")):
    print(f"\n{'='*50}")
    print(f"File: {audio_file.name}")
    print(f"{'='*50}")
    display(ipd.Audio(str(audio_file)))

## 7. Download Results

In [ ]:
# Download all output files as a zip
!zip -r outputs.zip outputs/

from google.colab import files
files.download("outputs.zip")

---

## Available Models

| Model | Category | Description |
|-------|----------|-------------|
| BS-RoFormer-SW | vocals | 6-stem separation (vocals, drums, bass, guitar, piano, other) |
| BS-RoFormer Vocals Resurrection | vocals | High-quality vocal extraction by unwa |
| BS-RoFormer Vocals Revive V3e | vocals | Latest vocal extraction by unwa |
| BS-RoFormer Instrumental Resurrection | instrumental | Instrumental extraction by unwa |
| BS-RoFormer De-Reverb | dereverb | Remove reverb from audio |

List all available models with:
```bash
bs-roformer-download --list-models
```

See the [model registry](https://github.com/openmirlab/bs-roformer-infer) for more options.